# Lektion 08 - Multi-Agent Designmønster


## Opsætning


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hvorfor Multi-Agent Systemer?

Virkelige opgaver som rejseplanlægning involverer mange forskellige slags ekspertise — logistik, lokal viden, budgettering og mere. En enkelt agent, der forsøger at håndtere alt, bliver hurtigt uhåndterlig.

Multi-agent systemer løser dette gennem **specialisering**: hver agent fokuserer på et ekspertiseområde og leverer resultater af højere kvalitet end en generalist. De forbedrer også **skalerbarhed** — du kan tilføje nye agenter (f.eks. en flyspecialist, en restaurantkritiker) uden at omskrive den eksisterende arbejdsgang. Agenterne komponeres sammen gennem en struktureret pipeline, der videresender kontekst fra den ene til den næste.


## Oprettelse af specialiserede agenter


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Oprettelse af et sekventielt workflow

`WorkflowBuilder` giver dig mulighed for at koble agenter sammen i en rettet graf. Her opretter vi en simpel to-trins pipeline: **TravelPlanner** udarbejder rejseplanen, og derefter gennemgår og forbedrer **TravelConcierge** den.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Tilføjelse af flere agenter til arbejdsgangen

En af de største fordele ved multi-agent mønsteret er, hvor nemt det er at udvide. Nedenfor tilføjer vi en **BudgetReviewer** agent, der tjekker planen mod rejsendes budget, markerer punkter, der kan få omkostningerne til at overskride grænsen, og foreslår pengebesparende alternativer. Arbejdsgangen kører nu tre agenter i rækkefølge:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Resumé

I denne lektion lærte du at:

1. **Oprette specialiserede agenter** — hver med en fokuseret rolle (planlægning, concierge, budgetgennemgang).
2. **Forbinde agenter i en sekventiel arbejdsgang** ved hjælp af `WorkflowBuilder` og `add_edge`.
3. **Stream output** fra en multi-agent pipeline, mens du sporer hvilken agent der taler.
4. **Udvide en arbejdsgang** ved at tilføje nye agenter til kæden uden at modificere de eksisterende.

Multi-agent designmønstret holder hver agent enkel, samtidig med at det producerer rigere og mere grundigt gennemgåede resultater, end nogen enkelt agent kunne opnå alene.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
